# Exp 39: Stacking with logistic regression

Card: [exp39_stacking_logreg.md](../../docs/neuroslope/experiments/exp39_stacking_logreg.md)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_recall_curve

from training.config import CLASSES, ROOT
from training.config import VIP
from training.runs import PROB_COLS
from analysis.thresholds import best_f1, plateau_f1
from training.ensemble import Ensemble, Stacking
from analysis.plots import save_figures, use_style

CONFIG_DIR = ROOT / "configs" / "models"
MODELS = {
    "exp26 ResNet-18": CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.yaml",
    "EfficientNet-B0": CONFIG_DIR / "exp32_efficientnet_b0_5fold" / "efficientnet_b0.yaml",
    "RegNetY-016": CONFIG_DIR / "exp33_regnety_016" / "regnety_016.yaml",
    "ConvNeXt-Tiny": CONFIG_DIR / "exp34_convnext_tiny" / "convnext_tiny.yaml",
}
CS = [1.0, 0.1]  # L2 strength: 1.0 scikit-learn default, 0.1 stronger
MARGIN = 0.05  # plateau: F1 >= max - MARGIN (exp37, exp38)
FIGURES = ROOT / "docs" / "figures"
PREFIX = "exp39_stacking_logreg"

# dataviz reference palette, light mode
INK, PEAK, MIDDLE = "#52514e", "#eb6834", "#2a78d6"
SERIES = {
    "plain mean": "#2a78d6",
    "stacking C = 1.0": "#eb6834",
    "stacking C = 0.1": "#1baf7a",
}
use_style()

## Plain mean and stacking


In [ ]:
oofs = Ensemble.member_oofs(MODELS.values())
labels, folds = oofs[0][CLASSES].to_numpy(), oofs[0]["fold"].to_numpy()
member_probs = np.stack([o[PROB_COLS].to_numpy() for o in oofs], axis=1)  # rows, members, classes
member_logits = Stacking.logit(member_probs)
mean_probs = member_probs.mean(axis=1)
# per class: features = members' logits of that class; each fold predicted by model fit on other folds
ensembles, weights = {"plain mean": mean_probs}, {}
for C in CS:
    ensembles[f"stacking C = {C}"], weights[C] = Stacking.cross_fold(member_logits, labels, folds, C, list(MODELS))

## Weights


In [ ]:
# coefficient per member logit and intercept; fold = held-out fold of that fit
for C, w in weights.items():
    print(f"C = {C}")
    display(w.round(2), w.groupby("class", sort=False).agg(["mean", "std"]).round(2))

## F1 against threshold


In [ ]:
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.6 * len(CLASSES), 3.6), sharey=True
)
for c, (cls, ax) in enumerate(zip(CLASSES, axes)):
    for name, probs in ensembles.items():
        precision, recall, thr = precision_recall_curve(labels[:, c], probs[:, c])
        f1 = (2 * precision * recall / (precision + recall + 1e-12))[:-1]
        ax.plot(thr, f1, color=SERIES[name], lw=2, label=name)
    ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
    ax.set(
        title=cls,
        xlabel="threshold",
        xlim=(0, 1),
        ylim=(0, 1.05),
        xticks=ticks,
        yticks=ticks,
    )
    ax.tick_params(labelleft=True, labelsize=8)
axes[0].set_ylabel("F1")
fig.suptitle(
    f"F1 against threshold, ensembles of {len(MODELS)} models, out-of-fold rows of folds 0–4",
    color=INK,
)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    ncols=3,
    bbox_to_anchor=(0.5, 0.93),
)
figures = {f"{PREFIX}_f1_curves.png": fig}
plt.show()

## Stacking C = 0.1: peak vs plateau


In [ ]:
# C = 0.1 stacking, VIP: peak vs plateau-middle threshold (exp38 layout)
probs = ensembles["stacking C = 0.1"]
peak, plateau = best_f1(labels, probs, CLASSES), plateau_f1(labels, probs, CLASSES, MARGIN)
fig, axes = plt.subplots(1, len(VIP), figsize=(4.2 * len(VIP), 3.6), sharey=True)
for cls, ax in zip(VIP, axes):
    c = CLASSES.index(cls)
    precision, recall, thr = precision_recall_curve(labels[:, c], probs[:, c])
    f1 = (2 * precision * recall / (precision + recall + 1e-12))[:-1]
    ax.axvspan(
        plateau.loc[cls, "low"],
        plateau.loc[cls, "high"],
        color=MIDDLE,
        alpha=0.12,
        lw=0,
        label=f"F1 within {MARGIN} of max",
    )
    ax.plot(thr, f1, color=INK, lw=1.5)
    best_t, mean_t = peak.loc[cls, "threshold"], plateau.loc[cls, "threshold"]
    ax.axvline(best_t, color=PEAK, lw=2, ls="--", label="best_t (peak)")
    ax.axvline(mean_t, color=MIDDLE, lw=2, label="mean_t (plateau middle)")
    # value labels at two heights, outward from the pair so they never cross a line; both left near right edge
    for t, label, y in [(best_t, "best_t", 0.22), (mean_t, "mean_t", 0.08)]:
        right = t == max(best_t, mean_t) and t < 0.8
        ax.text(
            t + 0.015 if right else min(best_t, mean_t) - 0.015,
            y,
            f"{label} {t:.3f}",
            ha="left" if right else "right",
            fontsize=8,
            color=INK,
        )
    ticks = [0, 0.2, 0.4, 0.6, 0.8, 1]
    ax.set(
        title=cls,
        xlabel="threshold",
        xlim=(0, 1),
        ylim=(0, 1.05),
        xticks=ticks,
        yticks=ticks,
    )
    ax.tick_params(labelleft=True, labelsize=8)
axes[0].set_ylabel("stacking C = 0.1\nF1")
fig.suptitle(
    "F1 against threshold, stacking C = 0.1, out-of-fold rows of folds 0–4", color=INK
)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    ncols=3,
    bbox_to_anchor=(0.5, 0.93),
)
figures[f"{PREFIX}_c01_plateau.png"] = fig
plt.show()

In [ ]:
save_figures(figures, FIGURES)